# 13 · PySpark DataFrame basics (Spark)

A **DataFrame** is a table spread over many machines (here: your laptop's CPU cores).
- **transformations** (`select`, `filter`, `withColumn`, `join`, ...) only **describe** the work: nothing runs yet
- **actions** (`show`, `count`, `collect`, `write`, ...) **run** it. This is **lazy evaluation**.

Import style used everywhere: `from pyspark.sql import functions as F, Window, types as T`.

**How to run:** Start a cell with `%%sql` to write Spark SQL. In a normal Python cell use the DataFrame API: the tables are ready as variables (`employees`, `orders`, ...). Put your answer in `res` and call `res.show()`.

Tables: see [tables_guide.md](../../cheatsheets/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`spark/practice/13_dataframe_basics.ipynb`](../../spark/practice/13_dataframe_basics.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "spark"))
from spark_setup import get_spark, load_tables
from pyspark.sql import functions as F, Window, types as T

spark = get_spark()
t = load_tables(spark)          # also makes SQL views with the same names
employees, departments = t["employees"], t["departments"]
customers, orders = t["customers"], t["orders"]
products, order_items = t["products"], t["order_items"]
user_logins = t["user_logins"]
%load_ext spark_setup

## Q1. Look at a DataFrame  ·  🟢 easy

For `employees`: print the schema, the list of column names, the column types, and the row count.

**How it works:** `printSchema`, `columns` and `dtypes` only read metadata; `count()` is an action that runs a job.

In [ ]:
employees.printSchema()
print(employees.columns)
print(employees.dtypes)
print(employees.count())

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- dept_id: integer (nullable = true)
 |-- manager_id: integer (nullable = true)
 |-- salary: decimal(10,2) (nullable = true)
 |-- hire_date: date (nullable = true)

['emp_id', 'name', 'dept_id', 'manager_id', 'salary', 'hire_date']
[('emp_id', 'int'), ('name', 'string'), ('dept_id', 'int'), ('manager_id', 'int'), ('salary', 'decimal(10,2)'), ('hire_date', 'date')]
30

**Why / traps:** `printSchema()` shows types and **nullable** (can it hold NULL). `count()` is an **action**: it runs a job.

## Q2. Select, compute, rename  ·  🟢 easy

Show `name`, `salary` and `new_salary` = salary × 1.10 rounded to 2 decimals. Then rename `name` to `employee_name`.

*Expected result: 30 rows · columns: `employee_name`, `salary`, `new_salary`*

**How it works:** `select` with an expression creates the new column; `withColumnRenamed` renames an existing one.

In [ ]:
%%sql
SELECT name AS employee_name, salary, ROUND(salary * 1.10, 2) AS new_salary
FROM employees

,employee_name,salary,new_salary
0,Alice Johnson,250000.00,275000.00
1,Bob Smith,150000.00,165000.00
2,Carol White,120000.00,132000.00
3,David Brown,120000.00,132000.00
4,Eva Green,110000.00,121000.00
5,Frank Moore,95000.00,104500.00
6,Grace Lee,95000.00,104500.00
7,Henry Clark,140000.00,154000.00
8,Irene Lewis,90000.00,99000.00
9,Jack Walker,90000.00,99000.00


**DataFrame API:**

In [ ]:
res = (employees
       .select("name", "salary", F.round(F.col("salary") * 1.10, 2).alias("new_salary"))
       .withColumnRenamed("name", "employee_name"))
res.show()

+-------------+---------+----------+
|employee_name|   salary|new_salary|
+-------------+---------+----------+
|Alice Johnson|250000.00|  275000.0|
|    Bob Smith|150000.00|  165000.0|
|  Carol White|120000.00|  132000.0|
|  David Brown|120000.00|  132000.0|
|    Eva Green|110000.00|  121000.0|
|  Frank Moore| 95000.00|  104500.0|
|    Grace Lee| 95000.00|  104500.0|
|  Henry Clark|140000.00|  154000.0|
|  Irene Lewis| 90000.00|   99000.0|
|  Jack Walker| 90000.00|   99000.0|
|   Karen Hall| 90000.00|   99000.0|
|    Leo Allen| 70000.00|   77000.0|
|    Mia Young| 65000.00|   71500.0|
|  Nathan King|130000.00|  143000.0|
| Olivia Scott| 85000.00|   93500.0|
|   Paul Adams| 85000.00|   93500.0|
|  Quinn Baker| 60000.00|   66000.0|
|Rachel Nelson|135000.00|  148500.0|
|   Sam Carter|100000.00|  110000.0|
|Tina Mitchell| 98000.00|  107800.0|
+-------------+---------+----------+
only showing top 20 rows

**Why / traps:** Columns can be strings (`"salary"`) or Column objects (`F.col("salary")`). You need Column objects for math and conditions.

## Q3. Add and cast columns  ·  🟢 easy

Add `hire_year` (int), `salary_double` (salary cast to double) and `hire_date_text` (hire_date as string). Use **one** call: `withColumns`.

*Expected result: 30 rows · columns: `name`, `hire_year`, `salary_double`, `hire_date_text`*

**How it works:** `withColumns` adds several columns in one step; `cast` changes the type.

In [ ]:
res = employees.withColumns({
    "hire_year": F.year("hire_date"),
    "salary_double": F.col("salary").cast("double"),
    "hire_date_text": F.col("hire_date").cast("string"),
}).select("name", "hire_year", "salary_double", "hire_date_text")
res.show()

+-------------+---------+-------------+--------------+
|         name|hire_year|salary_double|hire_date_text|
+-------------+---------+-------------+--------------+
|Alice Johnson|     2015|     250000.0|    2015-03-01|
|    Bob Smith|     2016|     150000.0|    2016-06-15|
|  Carol White|     2018|     120000.0|    2018-01-10|
|  David Brown|     2019|     120000.0|    2019-04-22|
|    Eva Green|     2020|     110000.0|    2020-07-01|
|  Frank Moore|     2021|      95000.0|    2021-09-13|
|    Grace Lee|     2022|      95000.0|    2022-02-28|
|  Henry Clark|     2016|     140000.0|    2016-11-05|
|  Irene Lewis|     2017|      90000.0|    2017-08-19|
|  Jack Walker|     2018|      90000.0|    2018-03-03|
|   Karen Hall|     2019|      90000.0|    2019-12-12|
|    Leo Allen|     2021|      70000.0|    2021-05-30|
|    Mia Young|     2023|      65000.0|    2023-01-16|
|  Nathan King|     2017|     130000.0|    2017-02-14|
| Olivia Scott|     2019|      85000.0|    2019-06-06|
|   Paul A

**Why / traps:** Many `withColumn` calls in a loop make the plan big and slow. `withColumns({...})` or one `select` is better.

## Q4. Filter with several conditions  ·  🟢 easy

Employees in departments 1 or 2, hired in **2018 or later**, whose name contains `a` (lower-case). Show `name`, `dept_id`, `hire_date`.

*Expected result: 8 rows · columns: `name`, `dept_id`, `hire_date`*

**How it works:** Combine conditions with `&` inside `filter`; `isin` = SQL IN, `contains` = LIKE '%a%'.

In [ ]:
%%sql
SELECT name, dept_id, hire_date
FROM employees
WHERE dept_id IN (1, 2) AND hire_date >= DATE '2018-01-01' AND name LIKE '%a%'

,name,dept_id,hire_date
0,Carol White,1,2018-01-10
1,David Brown,1,2019-04-22
2,Eva Green,1,2020-07-01
3,Frank Moore,1,2021-09-13
4,Grace Lee,1,2022-02-28
5,Jack Walker,2,2018-03-03
6,Karen Hall,2,2019-12-12
7,Mia Young,2,2023-01-16


**DataFrame API:**

In [ ]:
res = (employees
       .filter(F.col("dept_id").isin(1, 2)
               & (F.col("hire_date") >= "2018-01-01")
               & F.col("name").contains("a"))
       .select("name", "dept_id", "hire_date"))
res.show()

+-----------+-------+----------+
|       name|dept_id| hire_date|
+-----------+-------+----------+
|Carol White|      1|2018-01-10|
|David Brown|      1|2019-04-22|
|  Eva Green|      1|2020-07-01|
|Frank Moore|      1|2021-09-13|
|  Grace Lee|      1|2022-02-28|
|Jack Walker|      2|2018-03-03|
| Karen Hall|      2|2019-12-12|
|  Mia Young|      2|2023-01-16|
+-----------+-------+----------+

**Why / traps:** `filter` and `where` are the same. A filter can also be a SQL string: `.filter("dept_id IN (1, 2)")`.

## Q5. Handle NULLs: fillna, dropna  ·  🟢 easy

From `orders`: (1) `res` = orders with NULL amount replaced by 0 and NULL status by `unknown`. (2) print how many rows remain after dropping rows with NULL amount.

*Expected result: 108 rows · columns: `order_id`, `customer_id`, `order_date`, `amount`, `status`*

**How it works:** `fillna({col: value})` fills per column; `dropna(subset=[...])` drops rows with NULL in those columns only.

In [ ]:
res = orders.fillna({"amount": 0, "status": "unknown"})
print("rows without NULL amount:", orders.dropna(subset=["amount"]).count())
res.show()

rows without NULL amount: 102

+--------+-----------+----------+------+---------+
|order_id|customer_id|order_date|amount|   status|
+--------+-----------+----------+------+---------+
|    1001|         27|2023-03-15|135.00|completed|
|    1002|          7|2023-06-13| 45.00|completed|
|    1003|          3|2023-06-25| 70.00|completed|
|    1004|         19|2023-08-30| 10.00|completed|
|    1005|         27|2023-10-17|180.00|  pending|
|    1006|         30|2023-11-06|155.00|completed|
|    1007|         18|2023-11-16| 94.00|cancelled|
|    1008|         30|2023-11-23|  0.00|completed|
|    1009|         29|2023-12-23| 90.00|completed|
|    1010|         21|2023-12-25| 12.00|completed|
|    1011|         30|2024-01-03|120.00|cancelled|
|    1012|         21|2024-01-08| 82.00|completed|
|    1013|         29|2024-01-13| 45.00|completed|
|    1014|         27|2024-01-21| 80.00|completed|
|    1015|         20|2024-02-06|140.00|completed|
|    1016|         27|2024-02-07| 62.00|completed|
|    1017|         14|2024-02-2

**Why / traps:** `fillna` with a dict fills each column with its own value. `dropna()` with no subset drops a row if **any** column is NULL: usually too much.

## Q6. Create a DataFrame with an explicit schema  ·  🟡 medium

Build a DataFrame of 3 bonuses: `(1, 5000.0, '2025-01-31')`, `(8, 3000.0, '2025-01-31')`, `(99, 100.0, '2025-02-28')` with a **StructType** schema (emp_id int, bonus double, paid_on string), then cast paid_on to date.

*Expected result: 3 rows · columns: `emp_id`, `bonus`, `paid_on`*

**How it works:** Define the schema with StructType/StructField, build the DataFrame from Python tuples, then convert the text date with `to_date`.

In [ ]:
schema = T.StructType([
    T.StructField("emp_id", T.IntegerType(), nullable=False),
    T.StructField("bonus", T.DoubleType()),
    T.StructField("paid_on", T.StringType()),
])
bonuses = spark.createDataFrame([(1, 5000.0, "2025-01-31"), (8, 3000.0, "2025-01-31"),
                                 (99, 100.0, "2025-02-28")], schema)
res = bonuses.withColumn("paid_on", F.to_date("paid_on"))
res.show()

+------+------+----------+
|emp_id| bonus|   paid_on|
+------+------+----------+
|     1|5000.0|2025-01-31|
|     8|3000.0|2025-01-31|
|    99| 100.0|2025-02-28|
+------+------+----------+

**Why / traps:** An explicit schema is faster (no guessing) and safer (no wrong types). A short form also works: `"emp_id int, bonus double, paid_on string"`.

## Q7. Sort with NULLs last  ·  🟢 easy

Sort employees by `dept_id` ascending with the NULL dept_id **last**, then salary high → low. Show `name`, `dept_id`, `salary`.

*Expected result: 30 rows · columns: `name`, `dept_id`, `salary`*

**How it works:** `asc_nulls_last()` on the column moves NULLs to the end; then the other sort keys.

In [ ]:
%%sql
SELECT name, dept_id, salary
FROM employees
ORDER BY dept_id ASC NULLS LAST, salary DESC, name

,name,dept_id,salary
0,Alice Johnson,1.0,250000.00
1,Bob Smith,1.0,150000.00
2,Carol White,1.0,120000.00
3,David Brown,1.0,120000.00
4,Eva Green,1.0,110000.00
5,Frank Moore,1.0,95000.00
6,Grace Lee,1.0,95000.00
7,Henry Clark,2.0,140000.00
8,Irene Lewis,2.0,90000.00
9,Jack Walker,2.0,90000.00


**DataFrame API:**

In [ ]:
res = (employees.select("name", "dept_id", "salary")
       .orderBy(F.col("dept_id").asc_nulls_last(), F.desc("salary"), "name"))
res.show()

+-------------+-------+---------+
|         name|dept_id|   salary|
+-------------+-------+---------+
|Alice Johnson|      1|250000.00|
|    Bob Smith|      1|150000.00|
|  Carol White|      1|120000.00|
|  David Brown|      1|120000.00|
|    Eva Green|      1|110000.00|
|  Frank Moore|      1| 95000.00|
|    Grace Lee|      1| 95000.00|
|  Henry Clark|      2|140000.00|
|  Irene Lewis|      2| 90000.00|
|  Jack Walker|      2| 90000.00|
|   Karen Hall|      2| 90000.00|
|    Leo Allen|      2| 70000.00|
|    Mia Young|      2| 65000.00|
|  Nathan King|      3|130000.00|
| Olivia Scott|      3| 85000.00|
|   Paul Adams|      3| 85000.00|
|  Quinn Baker|      3| 60000.00|
|Rachel Nelson|      4|135000.00|
|   Sam Carter|      4|100000.00|
|Tina Mitchell|      4| 98000.00|
+-------------+-------+---------+
only showing top 20 rows

**Why / traps:** Spark (and BigQuery) put NULLs **first** for ASC by default; Oracle puts them **last**.

## Q8. distinct vs dropDuplicates  ·  🟢 easy

(1) Different `(customer_id, status)` pairs in orders: count them. (2) Keep **one row per customer** with `dropDuplicates(["customer_id"])` and count.

**How it works:** `distinct` compares whole rows; `dropDuplicates([cols])` compares only the listed columns and keeps one row each.

In [ ]:
print("pairs:", orders.select("customer_id", "status").distinct().count())
one_per_customer = orders.dropDuplicates(["customer_id"])
print("customers:", one_per_customer.count())

pairs: 54
customers: 32

**Why / traps:** `dropDuplicates(subset)` keeps **some** row per key: which one is not defined. Use a window with `row_number` to choose (topic 08).

## Q9. SQL inside the DataFrame API  ·  🟢 easy

Use `selectExpr` and `F.expr` to show `name`, `salary` and `band` = `CASE WHEN salary >= 100000 THEN 'senior' ELSE 'junior' END`.

*Expected result: 30 rows · columns: `name`, `salary`, `band`*

**How it works:** `selectExpr` takes SQL text for each column; `F.expr` puts SQL text inside a normal select.

In [ ]:
res = employees.selectExpr(
    "name", "salary",
    "CASE WHEN salary >= 100000 THEN 'senior' ELSE 'junior' END AS band")
# same thing with F.expr inside a normal select:
same = employees.select("name", "salary",
                        F.expr("CASE WHEN salary >= 100000 THEN 'senior' ELSE 'junior' END").alias("band"))
res.show()

+-------------+---------+------+
|         name|   salary|  band|
+-------------+---------+------+
|Alice Johnson|250000.00|senior|
|    Bob Smith|150000.00|senior|
|  Carol White|120000.00|senior|
|  David Brown|120000.00|senior|
|    Eva Green|110000.00|senior|
|  Frank Moore| 95000.00|junior|
|    Grace Lee| 95000.00|junior|
|  Henry Clark|140000.00|senior|
|  Irene Lewis| 90000.00|junior|
|  Jack Walker| 90000.00|junior|
|   Karen Hall| 90000.00|junior|
|    Leo Allen| 70000.00|junior|
|    Mia Young| 65000.00|junior|
|  Nathan King|130000.00|senior|
| Olivia Scott| 85000.00|junior|
|   Paul Adams| 85000.00|junior|
|  Quinn Baker| 60000.00|junior|
|Rachel Nelson|135000.00|senior|
|   Sam Carter|100000.00|senior|
|Tina Mitchell| 98000.00|junior|
+-------------+---------+------+
only showing top 20 rows

**Why / traps:** `expr` lets you mix SQL strings and Python. Handy for functions that are easier to write in SQL.

## Q10. Bring results to Python carefully  ·  🟢 easy

Get the **top 3** salaries as Python values with `take`, the first row with `first()`, and the employees table as **pandas** with only 5 rows.

**How it works:** `take(n)` and `first()` return Row objects to Python; `limit(n).toPandas()` returns a small pandas DataFrame.

In [ ]:
top3 = employees.orderBy(F.desc("salary")).select("name", "salary").take(3)
print(top3)
print(top3[0]["name"], top3[0].salary)
print(employees.first())
print(employees.limit(5).toPandas())

[Row(name='Alice Johnson', salary=Decimal('250000.00')), Row(name='Bob Smith', salary=Decimal('150000.00')), Row(name='Henry Clark', salary=Decimal('140000.00'))]
Alice Johnson 250000.00
Row(emp_id=1, name='Alice Johnson', dept_id=1, manager_id=None, salary=Decimal('250000.00'), hire_date=datetime.date(2015, 3, 1))
   emp_id           name  dept_id  manager_id     salary   hire_date
0       1  Alice Johnson        1         NaN  250000.00  2015-03-01
1       2      Bob Smith        1         1.0  150000.00  2016-06-15
2       3    Carol White        1         2.0  120000.00  2018-01-10
3       4    David Brown        1         2.0  120000.00  2019-04-22
4       5      Eva Green        1         2.0  110000.00  2020-07-01

**Why / traps:** `collect()`, `take()`, `toPandas()` bring data to the **driver** (one machine). On big data this causes out-of-memory errors. Always `limit` or aggregate first.

## Q11. Quick statistics  ·  🟢 easy

Show count, mean, stddev, min, max of `salary` with `describe`, and the 25%/50%/75% values with `summary`.

**How it works:** `describe()` gives count/mean/stddev/min/max; `summary()` can also give approximate percentiles.

In [ ]:
employees.describe("salary").show()
employees.select("salary").summary("25%", "50%", "75%").show()

+-------+-----------------+
|summary|           salary|
+-------+-----------------+
|  count|               30|
|   mean|    100366.666667|
| stddev|37133.01656935819|
|    min|         55000.00|
|    max|        250000.00|
+-------+-----------------+

+-------+--------+
|summary|  salary|
+-------+--------+
|    25%| 75000.0|
|    50%| 90000.0|
|    75%|115000.0|
+-------+--------+

**Why / traps:** Percentiles in `summary` are **approximate** (fast on big data).

## Q12. Rows per group, then filter (like HAVING)  ·  🟢 easy

Show `dept_id` and `n` for departments with **at least 5** employees, biggest first.

*Expected result: 3 rows · columns: `dept_id`, `n`*

**How it works:** groupBy + count, then `filter` on the count column works like SQL HAVING.

In [ ]:
%%sql
SELECT dept_id, COUNT(*) AS n
FROM employees
GROUP BY dept_id
HAVING COUNT(*) >= 5
ORDER BY n DESC, dept_id

,dept_id,n
0,1,7
1,2,6
2,6,5


**DataFrame API:**

In [ ]:
res = (employees.groupBy("dept_id").agg(F.count("*").alias("n"))
       .filter("n >= 5").orderBy(F.desc("n"), "dept_id"))
res.show()

+-------+---+
|dept_id|  n|
+-------+---+
|      1|  7|
|      2|  6|
|      6|  5|
+-------+---+